
# Liu2024 S-JEPA Spatial Topomap Diagnostics

This notebook generates cleaner topomap figures from one S-JEPA PreLocal run artifact folder.

It is designed for the Liu2024 artifact files:

- `diagnostic_summary.json`
- `fold_diagnostics.csv`
- `spatial_channel_importance_global.csv`
- `spatial_channel_importance_long.csv`
- `spatial_channel_importance_by_subject.csv`

Set `CONFIG["artifact_dir"]` to the run artifact path. The notebook will read the files from that directory and save publication/report-ready PNG and PDF figures.

The main plots are:

1. Final spatial importance topomaps for all folds, non-collapsed folds, and collapsed folds.
2. The same topomaps in raw shared scale and relative z-score scale.
3. Top-channel bar chart.
4. Optional subject-level topomap grid.
5. Optional absolute spatial-update topomaps if the artifact directory contains fold-level JSON with both initial and final spatial weights.


In [ ]:

# ---------------------------------------------------------------------
# 0. Config
# ---------------------------------------------------------------------

from pathlib import Path

CONFIG = {
    # Change this to the artifact folder you want to analyze.
    # Example:
    # "artifact_dir": Path("/home/vegorov/Repos/eeg_jepa_research/artifacts/liu2024-source-mat-sjepa-prelocal/20260526_1643_eb9de7fc"),
    "artifact_dir": Path("artifacts/liu2024-source-mat-sjepa-prelocal/20260526_1643_eb9de7fc_BEST_PRETRAINED_NEW/spatial_conv_analysis"),

    # Output folder. If None, the notebook creates artifact_dir / "topomap_diagnostics".
    "output_dir": None,

    # Label shown in figure titles and filenames.
    "run_label": "Liu2024 S-JEPA PreLocal",

    # Required artifact filenames.
    "diagnostic_summary_file": "diagnostic_summary.json",
    "fold_diagnostics_file": "fold_diagnostics.csv",
    "spatial_global_file": "spatial_channel_importance_global.csv",
    "spatial_long_file": "spatial_channel_importance_long.csv",
    "spatial_by_subject_file": "spatial_channel_importance_by_subject.csv",

    # Optional files. Only needed for true channel-wise spatial update topomaps.
    # spatial_channel_importance_long.csv gives final weights, but not per-channel initial-vs-final deltas.
    # If cv_results.json contains both spatial_conv_initial and spatial_conv, this notebook can plot update topomaps.
    "optional_weight_json_candidates": ["cv_results.json", "spatial_conv_weights_by_fold.json"],
    "try_optional_update_topomaps": False,

    # MNE montage. standard_1005 and standard_1020 both include Liu2024 channel names such as T3/T4/T5/T6.
    "montage_name": "standard_1005",
    "sfreq": 128.0,

    # Plot controls.
    "show_channel_names": True,
    "dpi": 300,
    "top_n_channels": 15,
    "topomap_contours": 6,

    # Subject grid is optional and can be useful for checking heterogeneity.
    # Leave empty to skip. Example: [18, 22, 30, 45, 19, 43]
    "subject_ids_for_grid": [],
    "subject_grid_value_col": "mean_abs_weight",
}

WORKING_DIR = Path.cwd().parent.parent
ARTIFACT_DIR = WORKING_DIR / CONFIG["artifact_dir"]
OUTPUT_DIR = CONFIG["output_dir"]
if OUTPUT_DIR is None:
    OUTPUT_DIR = ARTIFACT_DIR / "topomap_diagnostics"
else:
    OUTPUT_DIR = Path(OUTPUT_DIR).expanduser().resolve()

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Artifact dir: {ARTIFACT_DIR}")
print(f"Output dir:   {OUTPUT_DIR}")


In [ ]:

# ---------------------------------------------------------------------
# 1. Imports and path helpers
# ---------------------------------------------------------------------

import json
import math
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import colors as mcolors
import mne

plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "savefig.facecolor": "white",
    "font.size": 11,
    "axes.titlesize": 13,
    "axes.labelsize": 11,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
})


def artifact_path(filename: str, required: bool = True) -> Path | None:
    path = ARTIFACT_DIR / filename
    if required and not path.exists():
        raise FileNotFoundError(f"Required artifact not found: {path}")
    if not path.exists():
        return None
    return path


def clean_bool_series(series: pd.Series) -> pd.Series:
    """Convert bool/string/integer collapse flags into bool."""
    if series.dtype == bool:
        return series
    return series.astype(str).str.lower().map({
        "true": True,
        "1": True,
        "yes": True,
        "false": False,
        "0": False,
        "no": False,
    }).astype(bool)


def save_figure(fig, stem: str):
    png_path = OUTPUT_DIR / f"{stem}.png"
    pdf_path = OUTPUT_DIR / f"{stem}.pdf"
    fig.savefig(png_path, dpi=CONFIG["dpi"], bbox_inches="tight")
    fig.savefig(pdf_path, bbox_inches="tight")
    print(f"Saved PNG: {png_path}")
    print(f"Saved PDF: {pdf_path}")
    return png_path, pdf_path


In [ ]:

# ---------------------------------------------------------------------
# 2. Load run artifacts
# ---------------------------------------------------------------------

with open(artifact_path(CONFIG["diagnostic_summary_file"]), "r") as f:
    diagnostic_summary = json.load(f)

fold_df = pd.read_csv(artifact_path(CONFIG["fold_diagnostics_file"]))
spatial_global_df = pd.read_csv(artifact_path(CONFIG["spatial_global_file"]))
spatial_long_df = pd.read_csv(artifact_path(CONFIG["spatial_long_file"]))
spatial_by_subject_df = pd.read_csv(artifact_path(CONFIG["spatial_by_subject_file"]))

# Normalize dtypes.
for df in [fold_df, spatial_long_df]:
    if "collapse_flag" in df.columns:
        df["collapse_flag"] = clean_bool_series(df["collapse_flag"])

# Validate columns.
required_fold_cols = {
    "subject_id", "fold_id", "balanced_accuracy", "collapse_flag", "collapse_ratio",
    "spatial_delta_l2", "spatial_relative_delta", "mean_confidence", "mean_normalized_prediction_entropy",
}
required_long_cols = {
    "subject_id", "fold_id", "channel_index", "channel_name", "abs_mean_weight",
    "signed_mean_weight", "l2_weight", "balanced_accuracy", "collapse_flag",
}

missing_fold = required_fold_cols - set(fold_df.columns)
missing_long = required_long_cols - set(spatial_long_df.columns)
if missing_fold:
    raise ValueError(f"fold_diagnostics.csv missing columns: {sorted(missing_fold)}")
if missing_long:
    raise ValueError(f"spatial_channel_importance_long.csv missing columns: {sorted(missing_long)}")

print("Loaded artifacts:")
print(f"  diagnostic_summary: {diagnostic_summary}")
print(f"  fold_diagnostics: {fold_df.shape}")
print(f"  spatial_global: {spatial_global_df.shape}")
print(f"  spatial_long: {spatial_long_df.shape}")
print(f"  spatial_by_subject: {spatial_by_subject_df.shape}")

In [ ]:

# ---------------------------------------------------------------------
# 3. Collapsed vs non-collapsed fold summary
# ---------------------------------------------------------------------

def summarize_fold_group(name: str, df: pd.DataFrame) -> dict:
    return {
        "fold_group": name,
        "folds": len(df),
        "mean_accuracy_pct": 100.0 * df["accuracy"].mean() if "accuracy" in df.columns else np.nan,
        "mean_balanced_accuracy_pct": 100.0 * df["balanced_accuracy"].mean(),
        "mean_collapse_ratio": df["collapse_ratio"].mean(),
        "mean_spatial_delta_l2": df["spatial_delta_l2"].mean(),
        "mean_spatial_relative_delta": df["spatial_relative_delta"].mean(),
        "mean_confidence": df["mean_confidence"].mean(),
        "mean_normalized_prediction_entropy": df["mean_normalized_prediction_entropy"].mean(),
    }

summary_rows = [
    summarize_fold_group("All folds", fold_df),
    summarize_fold_group("Non-collapsed", fold_df[~fold_df["collapse_flag"]]),
    summarize_fold_group("Collapsed", fold_df[fold_df["collapse_flag"]]),
]
fold_group_summary = pd.DataFrame(summary_rows)

summary_path = OUTPUT_DIR / "collapsed_vs_noncollapsed_summary.csv"
fold_group_summary.to_csv(summary_path, index=False)
print(f"Saved summary CSV: {summary_path}")
fold_group_summary


In [ ]:

# ---------------------------------------------------------------------
# 4. Topomap helpers
# ---------------------------------------------------------------------

LIU2024_CHANNEL_ORDER = [
    "Fp1", "Fp2", "Fz", "F3", "F4", "F7", "F8", "FCz", "FC3", "FC4",
    "FT7", "FT8", "Cz", "C3", "C4", "T3", "T4", "CP3", "CP4", "TP7", "TP8",
    "Pz", "P3", "P4", "T5", "T6", "Oz", "O1", "O2",
]


def build_info_from_channel_names(channel_names, sfreq=128.0, montage_name="standard_1005"):
    montage = mne.channels.make_standard_montage(montage_name)
    montage_positions = montage.get_positions()["ch_pos"]

    valid_names = []
    valid_mask = []
    missing_names = []

    for ch in channel_names:
        ch = str(ch)
        has_pos = ch in montage_positions and np.all(np.isfinite(montage_positions[ch]))
        valid_mask.append(has_pos)
        if has_pos:
            valid_names.append(ch)
        else:
            missing_names.append(ch)

    if not valid_names:
        raise ValueError(f"No channel positions found for montage={montage_name}.")

    info = mne.create_info(ch_names=valid_names, sfreq=float(sfreq), ch_types="eeg")
    info.set_montage(montage, on_missing="ignore")

    if missing_names:
        print(f"Warning: excluded channels with missing montage positions: {missing_names}")

    return info, np.asarray(valid_mask, dtype=bool)


def order_channels(df: pd.DataFrame, channel_col="channel_name") -> pd.DataFrame:
    order = {ch: i for i, ch in enumerate(LIU2024_CHANNEL_ORDER)}
    out = df.copy()
    out["_order"] = out[channel_col].map(order)
    out = out.sort_values(["_order", channel_col]).drop(columns="_order")
    return out


def aggregate_spatial_long(df: pd.DataFrame, name: str) -> pd.DataFrame:
    if df.empty:
        raise ValueError(f"No folds available for group: {name}")

    agg = (
        df.groupby(["channel_index", "channel_name"], as_index=False)
        .agg(
            mean_abs_weight=("abs_mean_weight", "mean"),
            std_abs_weight=("abs_mean_weight", "std"),
            mean_signed_weight=("signed_mean_weight", "mean"),
            mean_l2_weight=("l2_weight", "mean"),
            n_fold_values=("abs_mean_weight", "size"),
            mean_balanced_accuracy=("balanced_accuracy", "mean"),
            mean_collapse_ratio=("collapse_ratio", "mean"),
            collapse_rate=("collapse_flag", "mean"),
            mean_spatial_delta_l2=("spatial_delta_l2", "mean"),
            mean_spatial_relative_delta=("spatial_relative_delta", "mean"),
        )
    )
    agg["group"] = name
    return order_channels(agg)


def transform_values(values: np.ndarray, mode: str) -> np.ndarray:
    values = np.asarray(values, dtype=float)
    if mode == "raw_shared":
        return values
    if mode == "relative_zscore":
        std = np.nanstd(values)
        if std < 1e-12:
            return np.zeros_like(values)
        return (values - np.nanmean(values)) / std
    if mode == "relative_percent":
        mean = np.nanmean(values)
        if abs(mean) < 1e-12:
            return np.zeros_like(values)
        return 100.0 * (values - mean) / mean
    raise ValueError(f"Unknown transform mode: {mode}")


def make_norm_and_cmap(all_values, mode: str, positive_only: bool = False):
    all_values = np.asarray(all_values, dtype=float)
    all_values = all_values[np.isfinite(all_values)]

    if mode == "raw_shared" or positive_only:
        vmin = 0.0 if np.nanmin(all_values) >= 0 else float(np.nanpercentile(all_values, 2))
        vmax = float(np.nanpercentile(all_values, 98))
        if not np.isfinite(vmax) or np.isclose(vmin, vmax):
            vmax = float(np.nanmax(all_values)) if len(all_values) else 1.0
        if vmax <= vmin:
            vmax = vmin + 1e-6
        return mcolors.Normalize(vmin=vmin, vmax=vmax), "viridis"

    # Symmetric scale for relative plots.
    p2, p98 = np.nanpercentile(all_values, [2, 98])
    lim = max(abs(float(p2)), abs(float(p98)))
    if not np.isfinite(lim) or lim < 1e-6:
        lim = max(1.0, float(np.nanmax(np.abs(all_values))))
    return mcolors.TwoSlopeNorm(vmin=-lim, vcenter=0.0, vmax=lim), "RdBu_r"


def plot_topomap_compat(data, info, ax, cmap, norm, names=None, contours=6):
    kwargs = dict(
        data=np.asarray(data, dtype=float),
        pos=info,
        axes=ax,
        show=False,
        cmap=cmap,
        contours=contours,
        names=names,
        sensors=True,
        outlines="head",
        extrapolate="head",
    )

    try:
        im, _ = mne.viz.plot_topomap(cnorm=norm, **kwargs)
    except TypeError:
        try:
            im, _ = mne.viz.plot_topomap(vlim=(norm.vmin, norm.vmax), **kwargs)
        except TypeError:
            im, _ = mne.viz.plot_topomap(vmin=norm.vmin, vmax=norm.vmax, **kwargs)

    if names is not None:
        for txt in ax.texts:
            txt.set_fontsize(8)
            txt.set_color("black")
    return im


In [ ]:

# ---------------------------------------------------------------------
# 5. Build all / non-collapsed / collapsed spatial tables
# ---------------------------------------------------------------------

spatial_groups = {
    "All folds": aggregate_spatial_long(spatial_long_df, "All folds"),
    "Non-collapsed": aggregate_spatial_long(spatial_long_df[~spatial_long_df["collapse_flag"]], "Non-collapsed"),
    "Collapsed": aggregate_spatial_long(spatial_long_df[spatial_long_df["collapse_flag"]], "Collapsed"),
}

for group_name, df in spatial_groups.items():
    out_path = OUTPUT_DIR / f"spatial_channel_importance_{group_name.lower().replace(' ', '_').replace('-', '_')}.csv"
    df.to_csv(out_path, index=False)
    print(f"{group_name}: {df.shape}, saved {out_path.name}")

spatial_groups["All folds"].head(10)


In [ ]:

 # ---------------------------------------------------------------------
# 6. Plot final spatial-importance topomaps
# ---------------------------------------------------------------------

def plot_spatial_importance_triptych(
    spatial_groups: dict,
    fold_group_summary: pd.DataFrame,
    value_col="mean_abs_weight",
    mode="raw_shared",
    stem="spatial_importance",
):
    prepared = {}
    transformed_values_all = []

    for group_name, df in spatial_groups.items():
        df = order_channels(df)
        channel_names = df["channel_name"].astype(str).tolist()
        raw_values = df[value_col].to_numpy(dtype=float)
        transformed_values = transform_values(raw_values, mode)

        info, valid_mask = build_info_from_channel_names(
            channel_names=channel_names,
            sfreq=CONFIG["sfreq"],
            montage_name=CONFIG["montage_name"],
        )

        prepared[group_name] = {
            "info": info,
            "values": transformed_values[valid_mask],
            "channel_names": [ch for ch, keep in zip(channel_names, valid_mask) if keep],
        }
        transformed_values_all.append(transformed_values[valid_mask])

    all_values = np.concatenate(transformed_values_all)
    norm, cmap = make_norm_and_cmap(all_values, mode=mode, positive_only=(mode == "raw_shared"))

    if mode == "raw_shared":
        cbar_label = value_col
    elif mode == "relative_zscore":
        cbar_label = f"within-group z-score of {value_col}"
    elif mode == "relative_percent":
        cbar_label = f"% deviation from group mean {value_col}"
    else:
        cbar_label = value_col

    fig, axes = plt.subplots(1, 3, figsize=(14.5, 4.6), constrained_layout=False)

    for ax, (group_name, item) in zip(axes, prepared.items()):
        group_summary = fold_group_summary[fold_group_summary["fold_group"] == group_name]
        if not group_summary.empty:
            folds = int(group_summary["folds"].iloc[0])
            mean_ba = group_summary["mean_balanced_accuracy_pct"].iloc[0]
            subtitle = f"{group_name}\nn={folds}, BA={mean_ba:.2f}%"
        else:
            subtitle = group_name

        names = item["channel_names"] if CONFIG["show_channel_names"] else None
        plot_topomap_compat(
            data=item["values"],
            info=item["info"],
            ax=ax,
            cmap=cmap,
            norm=norm,
            names=names,
            contours=CONFIG["topomap_contours"],
        )
        ax.set_title(subtitle, fontsize=12)

    fig.suptitle(f"{CONFIG['run_label']} - final spatial importance ({mode})", fontsize=15, y=0.98)
    sm = plt.cm.ScalarMappable(norm=norm, cmap=cmap)
    sm.set_array([])
    cbar = fig.colorbar(sm, ax=axes, fraction=0.026, pad=0.08)
    cbar.set_label(cbar_label, fontsize=11, labelpad=10)
    cbar.ax.tick_params(labelsize=9)

    fig.subplots_adjust(left=0.03, right=0.88, bottom=0.02, top=0.83, wspace=0.08)
    save_figure(fig, stem)
    return fig

fig_raw = plot_spatial_importance_triptych(
    spatial_groups,
    fold_group_summary,
    value_col="mean_abs_weight",
    mode="raw_shared",
    stem="topomap_final_spatial_importance_all_noncollapsed_collapsed_raw_shared",
)
plt.show()

fig_z = plot_spatial_importance_triptych(
    spatial_groups,
    fold_group_summary,
    value_col="mean_abs_weight",
    mode="relative_zscore",
    stem="topomap_final_spatial_importance_all_noncollapsed_collapsed_relative_zscore",
)
plt.show()


In [ ]:

# ---------------------------------------------------------------------
# 7. Top-channel bar chart for final spatial importance
# ---------------------------------------------------------------------

def plot_top_channels(df: pd.DataFrame, value_col="mean_abs_weight", top_n=15):
    top = df.sort_values(value_col, ascending=False).head(top_n).copy()
    top = top.iloc[::-1]

    fig, ax = plt.subplots(figsize=(8.0, max(4.5, 0.34 * top_n)))
    ax.barh(top["channel_name"], top[value_col])
    ax.set_xlabel(value_col)
    ax.set_ylabel("Channel")
    ax.set_title(f"{CONFIG['run_label']} - top {top_n} channels by {value_col}")
    ax.grid(axis="x", alpha=0.25)

    for y, val in enumerate(top[value_col]):
        ax.text(val, y, f" {val:.4f}", va="center", fontsize=9)

    fig.tight_layout()
    save_figure(fig, f"top_{top_n}_channels_by_{value_col}")
    return fig, top.iloc[::-1]

fig, top_channels = plot_top_channels(
    spatial_groups["All folds"],
    value_col="mean_abs_weight",
    top_n=CONFIG["top_n_channels"],
)
plt.show()

top_channels


In [ ]:

# ---------------------------------------------------------------------
# 8. Optional subject-level topomap grid
# ---------------------------------------------------------------------


def plot_subject_topomap_grid(subject_ids, by_subject_df, value_col="mean_abs_weight", mode="relative_zscore"):
    if not subject_ids:
        print("CONFIG['subject_ids_for_grid'] is empty. Skipping subject-level topomap grid.")
        return None

    subject_ids = [str(s) for s in subject_ids]
    by_subject = by_subject_df.copy()
    by_subject["subject_id"] = by_subject["subject_id"].astype(str)

    missing = [s for s in subject_ids if s not in set(by_subject["subject_id"])]
    if missing:
        raise ValueError(f"Subject IDs not found in spatial_by_subject file: {missing}")

    prepared = {}
    transformed_all = []

    for sid in subject_ids:
        df = order_channels(by_subject[by_subject["subject_id"] == sid])
        channel_names = df["channel_name"].astype(str).tolist()
        raw_values = df[value_col].to_numpy(dtype=float)
        transformed = transform_values(raw_values, mode)
        info, valid_mask = build_info_from_channel_names(channel_names, CONFIG["sfreq"], CONFIG["montage_name"])

        # Get subject mean BA if available.
        subj_folds = fold_df[fold_df["subject_id"].astype(str) == sid]
        mean_ba = 100.0 * subj_folds["balanced_accuracy"].mean() if not subj_folds.empty else np.nan
        collapse_rate = subj_folds["collapse_flag"].mean() if not subj_folds.empty else np.nan

        prepared[sid] = {
            "info": info,
            "values": transformed[valid_mask],
            "channel_names": [ch for ch, keep in zip(channel_names, valid_mask) if keep],
            "mean_ba": mean_ba,
            "collapse_rate": collapse_rate,
        }
        transformed_all.append(transformed[valid_mask])

    all_values = np.concatenate(transformed_all)
    norm, cmap = make_norm_and_cmap(all_values, mode=mode, positive_only=(mode == "raw_shared"))

    n = len(subject_ids)
    n_cols = min(4, n)
    n_rows = int(math.ceil(n / n_cols))
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(4.0 * n_cols, 3.8 * n_rows), squeeze=False)
    flat_axes = axes.ravel()

    for ax, sid in zip(flat_axes, subject_ids):
        item = prepared[sid]
        names = item["channel_names"] if CONFIG["show_channel_names"] else None
        plot_topomap_compat(item["values"], item["info"], ax, cmap, norm, names, CONFIG["topomap_contours"])
        ax.set_title(f"Subject {sid}\nBA={item['mean_ba']:.1f}%, collapse={item['collapse_rate']:.2f}", fontsize=11)

    for ax in flat_axes[n:]:
        ax.axis("off")

    fig.suptitle(f"{CONFIG['run_label']} - subject-level spatial importance ({mode})", fontsize=14, y=0.99)
    sm = plt.cm.ScalarMappable(norm=norm, cmap=cmap)
    sm.set_array([])
    cbar = fig.colorbar(sm, ax=flat_axes[:n], fraction=0.026, pad=0.03)
    cbar.set_label(value_col if mode == "raw_shared" else f"{mode} of {value_col}", fontsize=10)
    fig.subplots_adjust(left=0.03, right=0.90, bottom=0.02, top=0.88, wspace=0.05, hspace=0.22)
    save_figure(fig, f"subject_level_topomap_grid_{mode}")
    return fig

fig = plot_subject_topomap_grid(
    CONFIG["subject_ids_for_grid"],
    spatial_by_subject_df,
    value_col=CONFIG["subject_grid_value_col"],
    mode="relative_zscore",
)
if fig is not None:
    plt.show()


In [ ]:

# ---------------------------------------------------------------------
# 9. True absolute spatial-update topomaps
# ---------------------------------------------------------------------
# This cell keeps the existing CONFIG and artifact loading unchanged.
# It only adds spatial-update topomaps when a same-run JSON with both
# `spatial_conv_initial` and `spatial_conv` is available.
#
# The topomaps here show channel-wise absolute change in the spatial_conv
# layer, computed as mean(abs(final_weight - initial_weight)) across
# virtual spatial filters for each channel.


def candidate_update_json_paths():
    """Return possible same-run JSON paths without changing CONFIG or ARTIFACT_DIR."""
    filenames = CONFIG.get("optional_weight_json_candidates", ["cv_results.json", "spatial_conv_weights_by_fold.json"])
    bases = [ARTIFACT_DIR]

    # Many artifact layouts keep the spatial-analysis CSVs in a subfolder
    # and the full cv_results.json in the run folder one level above.
    for parent in [ARTIFACT_DIR.parent, ARTIFACT_DIR.parent.parent]:
        if parent not in bases:
            bases.append(parent)

    paths = []
    for base in bases:
        for filename in filenames:
            p = base / filename
            if p not in paths:
                paths.append(p)
    return paths


def json_has_initial_and_final_spatial_weights(json_path: Path) -> bool:
    """Quick check that a JSON file has records usable for true update topomaps."""
    if not json_path.exists():
        return False
    try:
        with open(json_path, "r") as f:
            records = json.load(f)
        if not isinstance(records, list) or len(records) == 0:
            return False
        first = records[0]
        return (
            isinstance(first, dict)
            and isinstance(first.get("spatial_conv_initial"), dict)
            and isinstance(first.get("spatial_conv"), dict)
            and "weight_matrix" in first["spatial_conv_initial"]
            and "weight_matrix" in first["spatial_conv"]
        )
    except Exception:
        return False


def find_update_weight_json():
    candidates = candidate_update_json_paths()
    for p in candidates:
        if json_has_initial_and_final_spatial_weights(p):
            print(f"Using spatial-update JSON: {p}")
            return p

    print("No same-run JSON with both spatial_conv_initial and spatial_conv was found.")
    print("Checked:")
    for p in candidates:
        print(f"  - {p}")
    return None


def extract_channel_update_long_from_json(json_path: Path) -> pd.DataFrame:
    with open(json_path, "r") as f:
        records = json.load(f)

    if not isinstance(records, list):
        raise ValueError(f"Expected list of fold records in {json_path}")

    rows = []
    skipped = 0

    for rec in records:
        initial = rec.get("spatial_conv_initial")
        final = rec.get("spatial_conv")
        if not isinstance(initial, dict) or not isinstance(final, dict):
            skipped += 1
            continue
        if "weight_matrix" not in initial or "weight_matrix" not in final:
            skipped += 1
            continue

        init_w = np.asarray(initial["weight_matrix"], dtype=float)
        final_w = np.asarray(final["weight_matrix"], dtype=float)
        if init_w.shape != final_w.shape:
            skipped += 1
            continue

        channel_names = final.get("channel_names") or initial.get("channel_names")
        if channel_names is None or len(channel_names) != init_w.shape[1]:
            skipped += 1
            continue

        delta = final_w - init_w
        abs_update_by_channel = np.mean(np.abs(delta), axis=0)
        signed_update_by_channel = np.mean(delta, axis=0)
        l2_update_by_channel = np.sqrt(np.sum(delta ** 2, axis=0))

        collapse_diagnostics = rec.get("collapse_diagnostics", {}) or {}
        probability_diagnostics = rec.get("probability_diagnostics", {}) or {}

        for idx, ch in enumerate(channel_names):
            rows.append({
                "subject_id": rec.get("subject_id"),
                "fold_id": rec.get("fold_id"),
                "channel_index": idx,
                "channel_name": ch,
                "abs_spatial_update": abs_update_by_channel[idx],
                "signed_spatial_update": signed_update_by_channel[idx],
                "l2_spatial_update": l2_update_by_channel[idx],
                "balanced_accuracy": rec.get("balanced_accuracy", np.nan),
                "accuracy": rec.get("accuracy", np.nan),
                "collapse_flag": collapse_diagnostics.get("collapse_flag", np.nan),
                "collapse_ratio": collapse_diagnostics.get("collapse_ratio", np.nan),
                "mean_confidence": probability_diagnostics.get("mean_confidence", np.nan),
                "mean_normalized_prediction_entropy": probability_diagnostics.get("mean_normalized_prediction_entropy", np.nan),
            })

    if not rows:
        raise ValueError(
            f"Could not extract channel-wise spatial updates from {json_path}. "
            "The file may contain final spatial weights only, but not initial weights."
        )

    out = pd.DataFrame(rows)
    out["collapse_flag"] = clean_bool_series(out["collapse_flag"])
    print(f"Extracted spatial updates: {out.shape[0]} channel rows from {json_path}")
    if skipped:
        print(f"Skipped records without usable initial/final weights: {skipped}")
    return out


def aggregate_update_long(df: pd.DataFrame, name: str) -> pd.DataFrame:
    if df.empty:
        raise ValueError(f"No folds available for group: {name}")

    agg = (
        df.groupby(["channel_index", "channel_name"], as_index=False)
        .agg(
            mean_abs_update=("abs_spatial_update", "mean"),
            std_abs_update=("abs_spatial_update", "std"),
            mean_signed_update=("signed_spatial_update", "mean"),
            mean_l2_update=("l2_spatial_update", "mean"),
            n_fold_values=("abs_spatial_update", "size"),
            mean_balanced_accuracy=("balanced_accuracy", "mean"),
            mean_collapse_ratio=("collapse_ratio", "mean"),
            collapse_rate=("collapse_flag", "mean"),
            mean_confidence=("mean_confidence", "mean"),
            mean_normalized_prediction_entropy=("mean_normalized_prediction_entropy", "mean"),
        )
    )
    agg["group"] = name
    return order_channels(agg)


def plot_spatial_update_triptych(
    update_groups: dict,
    fold_group_summary: pd.DataFrame,
    value_col="mean_abs_update",
    mode="raw_shared",
    stem="spatial_update",
):
    prepared = {}
    transformed_values_all = []

    for group_name, df in update_groups.items():
        df = order_channels(df)
        channel_names = df["channel_name"].astype(str).tolist()
        raw_values = df[value_col].to_numpy(dtype=float)
        transformed_values = transform_values(raw_values, mode)

        info, valid_mask = build_info_from_channel_names(
            channel_names=channel_names,
            sfreq=CONFIG["sfreq"],
            montage_name=CONFIG["montage_name"],
        )

        prepared[group_name] = {
            "info": info,
            "values": transformed_values[valid_mask],
            "channel_names": [ch for ch, keep in zip(channel_names, valid_mask) if keep],
        }
        transformed_values_all.append(transformed_values[valid_mask])

    all_values = np.concatenate(transformed_values_all)
    norm, cmap = make_norm_and_cmap(all_values, mode=mode, positive_only=(mode == "raw_shared"))

    if mode == "raw_shared":
        cbar_label = value_col
    elif mode == "relative_zscore":
        cbar_label = f"within-group z-score of {value_col}"
    elif mode == "relative_percent":
        cbar_label = f"% deviation from group mean {value_col}"
    else:
        cbar_label = value_col

    # Same figure layout and color-bar placement as the final spatial-importance topomap.
    fig, axes = plt.subplots(1, 3, figsize=(14.5, 4.6), constrained_layout=False)

    for ax, (group_name, item) in zip(axes, prepared.items()):
        group_summary = fold_group_summary[fold_group_summary["fold_group"] == group_name]
        if not group_summary.empty:
            folds = int(group_summary["folds"].iloc[0])
            mean_ba = group_summary["mean_balanced_accuracy_pct"].iloc[0]
            subtitle = f"{group_name}\nn={folds}, BA={mean_ba:.2f}%"
        else:
            subtitle = group_name

        names = item["channel_names"] if CONFIG["show_channel_names"] else None
        plot_topomap_compat(
            data=item["values"],
            info=item["info"],
            ax=ax,
            cmap=cmap,
            norm=norm,
            names=names,
            contours=CONFIG["topomap_contours"],
        )
        ax.set_title(subtitle, fontsize=12)

    fig.suptitle(f"{CONFIG['run_label']} - absolute spatial update ({mode})", fontsize=15, y=0.98)
    sm = plt.cm.ScalarMappable(norm=norm, cmap=cmap)
    sm.set_array([])
    cbar = fig.colorbar(sm, ax=axes, fraction=0.026, pad=0.08)
    cbar.set_label(cbar_label, fontsize=11, labelpad=10)
    cbar.ax.tick_params(labelsize=9)

    fig.subplots_adjust(left=0.03, right=0.88, bottom=0.02, top=0.83, wspace=0.08)
    save_figure(fig, stem)
    return fig


update_json_path = find_update_weight_json()

if update_json_path is not None:
    update_long_df = extract_channel_update_long_from_json(update_json_path)

    update_long_path = OUTPUT_DIR / "spatial_channel_update_long_from_json.csv"
    update_long_df.to_csv(update_long_path, index=False)
    print(f"Saved update CSV: {update_long_path}")

    update_groups = {
        "All folds": aggregate_update_long(update_long_df, "All folds"),
        "Non-collapsed": aggregate_update_long(update_long_df[~update_long_df["collapse_flag"]], "Non-collapsed"),
        "Collapsed": aggregate_update_long(update_long_df[update_long_df["collapse_flag"]], "Collapsed"),
    }

    for group_name, df in update_groups.items():
        out_path = OUTPUT_DIR / f"spatial_channel_update_{group_name.lower().replace(' ', '_').replace('-', '_')}.csv"
        df.to_csv(out_path, index=False)
        print(f"{group_name}: {df.shape}, saved {out_path.name}")

    fig_update_raw = plot_spatial_update_triptych(
        update_groups,
        fold_group_summary,
        value_col="mean_abs_update",
        mode="raw_shared",
        stem="topomap_absolute_spatial_update_all_noncollapsed_collapsed_raw_shared",
    )
    plt.show()

    fig_update_z = plot_spatial_update_triptych(
        update_groups,
        fold_group_summary,
        value_col="mean_abs_update",
        mode="relative_zscore",
        stem="topomap_absolute_spatial_update_all_noncollapsed_collapsed_relative_zscore",
    )
    plt.show()
else:
    print("Spatial update topomaps were not generated because no usable initial/final spatial-weight JSON was found.")



## Notes for report use

- The final spatial-importance topomaps show the mean absolute weights of the learned `spatial_conv` projection layer.
- These maps are not raw EEG power maps and should not be interpreted as direct physiological localization.
- The all/non-collapsed/collapsed split is computed from `collapse_flag` in `fold_diagnostics.csv` and `spatial_channel_importance_long.csv`.
- If update topomaps are skipped, it means the artifact folder did not contain fold-level records with both initial and final spatial weights. In that case, the notebook still correctly generates final spatial-importance topomaps from the attached CSV files.
